# 01 — EDA dữ liệu bước 0, tầng 3 và tầng 4
Đọc **split chuẩn đã tạo**, không chia dữ liệu lại. Thống kê số dòng/nhãn/nguồn trên toàn CSV theo chunk;
biểu đồ và kiểm tra nội dung dùng mẫu ngẫu nhiên tối đa 5.000 dòng mỗi file để giới hạn RAM.
Mẫu EDA không phải tập để train hoặc công bố metric.

Bản chi tiết: thống kê chất lượng toàn file, nguồn–nhãn, phân phối độ dài, trùng lặp, nhóm nguồn, mức khó negative, lexical drift, tín hiệu tắt và audit toàn bộ tùy chọn. Mọi biểu đồ được lưu vào REPORT_DIR.

## Môi trường và đường dẫn
Chọn kernel Python đã cài dependency của repo. Notebook không tự cài/upgrade PyTorch.
Chạy các ô từ trên xuống. Có thể cấu hình `CATEX_PROJECT_ROOT`, `CATEX_DATA_DIR`, `CATEX_OUTPUT_ROOT` hoặc sửa ô dưới.
Không đưa `.env` lên notebook. Các notebook này không gọi Web Search hay LLM.


In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import display

# Local: mở notebook từ repo hoặc notebooks/. Kaggle: sửa PROJECT_ROOT trỏ tới code đã giải nén.
PROJECT_ROOT = os.environ.get("CATEX_PROJECT_ROOT", "")
if PROJECT_ROOT:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "src/models/runtime.py").is_file()), None)
if ROOT is None or not (ROOT / "src/models/runtime.py").is_file():
    raise FileNotFoundError("Đặt PROJECT_ROOT thành thư mục chứa src/ và notebooks/.")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

# Kaggle: sửa thành đường dẫn thư mục split hiển thị trong Input; không ghi vào Input.
DATA_DIR = Path(os.environ.get("CATEX_DATA_DIR", str(ROOT / "data/splits"))).resolve()
DEFAULT_OUTPUT = Path("/kaggle/working/catex") if Path("/kaggle").exists() else ROOT
OUTPUT_ROOT = Path(os.environ.get("CATEX_OUTPUT_ROOT", str(DEFAULT_OUTPUT))).resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
from notebooks.workflow import inspect_csv, run_script, read_json, require_checkpoint
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print("Code:", ROOT, "\nData:", DATA_DIR, "\nOutput:", OUTPUT_ROOT)

from notebooks.eda_tools import profile_csv, text_features, lexical_pairs, normalize_text, disk_overlap_audit


In [ ]:
SAMPLE_SIZE = 5000
SEED = 42
REPORT_DIR = OUTPUT_ROOT / "evaluation/eda_results/notebook_01"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
specs = {"tang0": ["sentence"], "triplet_data": ["anchor","positive","negative"],
         "pairwise_labeled": ["sentence_1","sentence_2"]}
samples, summaries = {}, []
for prefix, columns in specs.items():
    for split in ("train","val","test"):
        path = DATA_DIR / f"{prefix}_{split}.csv"
        summary, sample = profile_csv(path, columns, SAMPLE_SIZE, SEED)
        summaries.append(summary); samples[(prefix,split)] = sample
summary_table = pd.DataFrame(summaries)
display(summary_table)
summary_table.to_json(REPORT_DIR / "counts.json", orient="records", force_ascii=False, indent=2)


## Kiểm tra audit và rò rỉ
Báo cáo dưới đây là **audit đã lưu khi chuẩn bị dữ liệu**, không phải kiểm tra mới cho những file có thể đã bị sửa.
Kiểm tra mẫu bên dưới chỉ phát hiện một số lỗi; không có overlap trong mẫu không chứng minh toàn bộ tập sạch.
Nếu thay đổi dữ liệu, chạy lại `python src/data_prep/audit_datasets.py <thư-mục-splits>` trên bản dữ liệu có quyền ghi.


In [ ]:
audit = read_json(DATA_DIR / "audit_report.json")
display(audit)
def normalized(text):
    import unicodedata
    return " ".join(unicodedata.normalize("NFC", text).split())
text_sets, group_sets = {}, {}
for split in ("train","val","test"):
    texts, groups = set(), set()
    for prefix, columns in specs.items():
        sample = samples[(prefix,split)]
        for column in columns: texts.update(sample[column].map(normalized))
        for column in ("group_id","candidate_group_id","negative_group_id"):
            if column in sample: groups.update(sample[column].dropna())
    text_sets[split], group_sets[split] = texts, groups
for a,b in [("train","val"),("train","test"),("val","test")]:
    counts = {"text_overlap_in_sample":len(text_sets[a] & text_sets[b]),
              "group_overlap_in_sample":len(group_sets[a] & group_sets[b])}
    print(a,b,counts)
    assert not any(counts.values()), "Phát hiện rò rỉ; dừng trước khi train."


## Bước 0 — nhãn AI-touched và độ dài

In [ ]:
single = samples[("tang0","train")].copy()
single["words"] = single.sentence.str.split().str.len()
display(single.groupby("label").words.describe())
display(pd.crosstab(single.source,single.label))
fig,ax=plt.subplots(figsize=(8,4))
for label,part in single.groupby("label"):
    ax.hist(part.words,bins=35,alpha=.5,label=f"label={label}")
ax.set(xlabel="Số từ",ylabel="Số câu trong mẫu",title="Bước 0: độ dài theo nhãn")
ax.legend();fig.tight_layout();fig.savefig(REPORT_DIR / "step0_lengths.png");plt.show()


## Tầng 3 — kiểm tra triplet
TF-IDF dưới đây chỉ là phép khảo sát từ vựng, fit trên **mẫu train**. Negative có cosine cao hơn positive
không tự động là nhãn sai; cần đọc nội dung. Vectorizer này không thay thế artifact tầng 1.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
triplets=samples[("triplet_data","train")]
assert (triplets.group_id != triplets.negative_group_id).all()
assert (triplets.anchor.map(normalized) != triplets.negative.map(normalized)).all()
v=TfidfVectorizer().fit(pd.concat([triplets.anchor,triplets.positive,triplets.negative]))
a=v.transform(triplets.anchor)
ap=np.asarray(a.multiply(v.transform(triplets.positive)).sum(axis=1)).ravel()
an=np.asarray(a.multiply(v.transform(triplets.negative)).sum(axis=1)).ravel()
print({"mean_AP":float(ap.mean()),"mean_AN":float(an.mean()),"fraction_AN_ge_AP":float((an>=ap).mean())})
fig,ax=plt.subplots(figsize=(8,4))
ax.hist(ap,bins=30,alpha=.5,label="Anchor–Positive");ax.hist(an,bins=30,alpha=.5,label="Anchor–Negative")
ax.legend();ax.set_xlabel("TF-IDF cosine");fig.savefig(REPORT_DIR / "triplet_similarity.png");plt.show()
display(triplets.sample(min(5,len(triplets)),random_state=SEED))


## Tầng 4 — cặp nguồn/câu đối chiếu

In [ ]:
pairs=samples[("pairwise_labeled","train")]
same_group=pairs.group_id.eq(pairs.candidate_group_id)
assert same_group.eq(pairs.label.eq(1)).all()
negative=pairs[pairs.label.eq(0)]
assert not negative.sentence_1.map(normalized).eq(negative.sentence_2.map(normalized)).any()
display(pairs.groupby("label").size().rename("sample_count"))
display(pairs.sample(min(6,len(pairs)),random_state=SEED))
fig,ax=plt.subplots(figsize=(8,4))
for column in ("sentence_1","sentence_2"):
    ax.hist(pairs[column].str.split().str.len(),bins=35,alpha=.5,label=column)
ax.legend();ax.set_xlabel("Số từ");fig.savefig(REPORT_DIR / "pair_lengths.png");plt.show()


## 5. Chất lượng toàn bộ dữ liệu và quy mô split
Các số đếm dưới đây quét **toàn bộ 9 CSV**, gồm nhãn không hợp lệ, text thiếu/rỗng,
negative trùng anchor/positive và nhóm nguồn sai. `blank_including_missing` bao gồm NaN, không cộng hai cột này thành tổng lỗi.
Phân vị độ dài tính từ histogram số từ toàn file, không ước lượng từ mẫu. Số từ không phải số token của Transformer.

In [ ]:
quality=[];length_rows=[]
for report in summaries:
    quality.append({"file":report["file"],"rows":report["rows"],
        "missing_cells":sum(report["missing"].values()),
        "blank_cells_including_missing":sum(report["blank_including_missing"].values()),
        **{k:report.get(k,0) for k in ["invalid_labels","invalid_negative_group","negative_equals_anchor_or_positive","invalid_pair_provenance","identical_negative_pair"]}})
    length_rows.extend({"file":report["file"],**row} for row in report["word_lengths"])
quality=pd.DataFrame(quality);length_table=pd.DataFrame(length_rows)
display(quality);display(length_table)
quality.to_csv(REPORT_DIR / "full_quality.csv",index=False)
length_table.to_csv(REPORT_DIR / "full_word_lengths.csv",index=False)
fig,ax=plt.subplots(figsize=(10,4))
quality.set_index("file").rows.plot.bar(ax=ax)
ax.set_ylabel("Số dòng toàn file");fig.tight_layout();fig.savefig(REPORT_DIR / "split_sizes.png");plt.show()


## 6. Quan hệ nguồn–nhãn và baseline lớp đa số
Một nguồn chỉ chứa lớp 1 có thể tạo dấu hiệu nhận diện nguồn thay vì dấu hiệu AI. Không tự cân bằng thành 50/50:
cần giữ mục tiêu thí nghiệm và xem precision/recall. Baseline dưới đây chọn lớp đa số từ train, sau đó chỉ kiểm tra trên validation.

In [ ]:
tables={}
for split in ("train","val","test"):
    report=next(r for r in summaries if r["file"]==f"tang0_{split}.csv")
    table=pd.DataFrame(report["source_labels"]).pivot_table(index="source",columns="label",values="rows",fill_value=0)
    tables[split]=table
    print(split);display(table);display(table.div(table.sum(axis=1),axis=0).rename(columns={0:"P(label0|source)",1:"P(label1|source)"}))
    table.to_csv(REPORT_DIR / f"source_label_{split}.csv")
fig,axes=plt.subplots(1,2,figsize=(12,4))
tables["train"].plot.bar(stacked=True,ax=axes[0],title="Train: số dòng theo nguồn/nhãn")
tables["train"].div(tables["train"].sum(axis=1),axis=0).plot.bar(stacked=True,ax=axes[1],title="Train: tỷ lệ trong từng nguồn")
fig.tight_layout();fig.savefig(REPORT_DIR / "source_label.png");plt.show()
majority=int(tables["train"].sum().idxmax())
val_counts=tables["val"].sum()
print({"train_majority_label":majority,"validation_majority_accuracy":float(val_counts.get(majority,0)/val_counts.sum())})


## 7. Phân phối độ dài và dấu hiệu văn phong — mẫu
Phân vị, boxplot và ECDF giúp thấy đuôi dài thay vì chỉ so sánh trung bình.
Các tỷ lệ ký tự dưới đây là mô tả; chữ hoa `[A-Z]` chỉ tính Latin ASCII, không phải bộ nhận diện ngôn ngữ/AI.

In [ ]:
feature_frames=[]
for split in ("train","val","test"):
    sample=samples[("tang0",split)]
    features=text_features(sample.sentence)
    features["label"]=sample.label.to_numpy();features["source"]=sample.source.to_numpy();features["split"]=split
    feature_frames.append(features)
features=pd.concat(feature_frames,ignore_index=True)
display(features.groupby(["split","label"])[["words","characters","digit_ratio","punctuation_ratio","uppercase_ratio","unique_word_ratio"]].agg(["mean","median","std"]))
features.to_csv(REPORT_DIR / "sample_text_features.csv",index=False)
fig,axes=plt.subplots(1,2,figsize=(12,4))
for (split,label),part in features.groupby(["split","label"]):
    x=np.sort(part.words);axes[0].plot(x,np.arange(1,len(x)+1)/len(x),label=f"{split}/{label}")
axes[0].set(xlabel="Số từ",ylabel="ECDF",title="Độ dài theo split và nhãn");axes[0].legend(fontsize=8)
features[features.split.eq("train")].boxplot(column="words",by="label",ax=axes[1],showfliers=False)
fig.suptitle("");fig.tight_layout();fig.savefig(REPORT_DIR / "sample_length_ecdf.png");plt.show()
# Xuất ví dụ ngắn/dài để xem bằng mắt, không tự loại bỏ.
review=single.assign(words=single.sentence.str.split().str.len()).sort_values("words")
display(pd.concat([review.head(5),review.tail(5)]))


## 8. Trùng lặp nội bộ và nhãn mâu thuẫn — mẫu
Đếm dòng trùng vượt lần xuất hiện đầu tiên. Hai quy tắc chuẩn hóa được tách rõ:
NFC + gộp khoảng trắng giống audit; casefold chỉ là khảo sát bổ sung. Không tự xóa câu trùng vì khác ngữ cảnh/nguồn có thể cần kiểm tra.

In [ ]:
duplicates=[]
for (prefix,split),sample in samples.items():
    cols=specs[prefix]
    normalized_frame=sample[cols].apply(lambda col:col.map(normalize_text))
    row={"view":prefix,"split":split,"sample_rows":len(sample),
         "duplicate_text_rows":int(normalized_frame.duplicated().sum()),
         "duplicate_casefold_rows":int(normalized_frame.apply(lambda c:c.str.casefold()).duplicated().sum())}
    if prefix=="tang0":
        tmp=pd.DataFrame({"text":normalized_frame.sentence,"label":sample.label})
        row["conflicting_text_labels"]=int(tmp.groupby("text").label.nunique().gt(1).sum())
    duplicates.append(row)
duplicates=pd.DataFrame(duplicates);display(duplicates)
duplicates.to_csv(REPORT_DIR / "sample_duplicates.csv",index=False)
conflict_path=DATA_DIR / "label_conflicts.csv"
if conflict_path.exists():
    conflict_report,conflict_sample=profile_csv(conflict_path,["sentence"],sample_size=10,seed=SEED)
    print("Số dòng trong vùng cách ly nhãn:",conflict_report["rows"],"(không đồng nghĩa số text duy nhất)")
    display(conflict_sample[["sentence","label","source","split"]])
else:print("Không có file label_conflicts.csv; không suy ra rằng không có conflict.")


## 9. Kích thước nhóm nguồn — toàn source_manifest
Một group có thể gom nhiều nguồn liên quan. Thống kê dưới đây đếm **record trong manifest**, không phải số câu train.
Nhóm lớn có thể làm tỷ lệ số dòng lệch khỏi tỷ lệ group dù chia đúng; không tự tách group để ép tỷ lệ.

In [ ]:
from collections import Counter
group_sizes=Counter()
manifest=DATA_DIR / "source_manifest.csv"
if manifest.exists():
    for chunk in pd.read_csv(manifest,usecols=["group_id","split"],dtype=str,chunksize=100000):
        group_sizes.update(chunk.groupby(["split","group_id"]).size().to_dict())
    groups=pd.DataFrame([{"split":s,"group_id":g,"source_records":count} for (s,g),count in group_sizes.items()])
    display(groups.groupby("split").source_records.describe(percentiles=[.5,.9,.95,.99]))
    display(groups.nlargest(10,"source_records"))
    groups.groupby("split").source_records.describe().to_csv(REPORT_DIR / "group_sizes.csv")
    fig,ax=plt.subplots(figsize=(8,4))
    for split,part in groups.groupby("split"):ax.hist(np.log1p(part.source_records),bins=30,alpha=.4,label=split)
    ax.set_xlabel("log(1 + số source record/group)");ax.legend();fig.tight_layout();fig.savefig(REPORT_DIR / "group_sizes.png");plt.show()
else:print("Thiếu source_manifest.csv; bỏ qua phân tích group toàn bộ.")


## 10. Mức khó negative và drift từ vựng
Dùng cùng vectorizer `v` đã fit trên mẫu triplet **train** để chấm cả train/val/test.
So sánh `lexical_hard` với `random`, cosine, Jaccard từ, tỷ lệ độ dài và gap AP−AN.
Đây là mô tả độ giống từ vựng, chưa phải đánh giá chất lượng embedding của tầng 3.

In [ ]:
triplet_scores=[];pair_scores=[]
for split in ("train","val","test"):
    frame=samples[("triplet_data",split)]
    pos=lexical_pairs(frame,"anchor","positive",v);neg=lexical_pairs(frame,"anchor","negative",v)
    score=pd.DataFrame({"split":split,"negative_type":frame.negative_type,
        "ap_cosine":pos.cosine,"an_cosine":neg.cosine,"gap":pos.cosine-neg.cosine,
        "ap_jaccard":pos.word_jaccard,"an_jaccard":neg.word_jaccard,
        "positive_length_ratio":pos.length_ratio,"negative_length_ratio":neg.length_ratio})
    triplet_scores.append(score)
    pairs_frame=samples[("pairwise_labeled",split)]
    paired=lexical_pairs(pairs_frame,"sentence_1","sentence_2",v)
    paired["split"]=split;paired["label"]=pairs_frame.label;paired["pair_type"]=pairs_frame.pair_type
    pair_scores.append(paired)
triplet_scores=pd.concat(triplet_scores,ignore_index=True);pair_scores=pd.concat(pair_scores,ignore_index=True)
display(triplet_scores.groupby(["split","negative_type"])[["ap_cosine","an_cosine","gap","an_jaccard"]].agg(["mean","median"]))
display(pair_scores.groupby(["split","label"])[["cosine","word_jaccard","length_ratio","exact_equal"]].mean())
triplet_scores.to_csv(REPORT_DIR / "sample_triplet_scores.csv",index=False)
pair_scores.to_csv(REPORT_DIR / "sample_pair_scores.csv",index=False)
fig,axes=plt.subplots(1,2,figsize=(12,4))
for kind,part in triplet_scores[triplet_scores.split.eq("train")].groupby("negative_type"):
    axes[0].hist(part.gap,bins=35,alpha=.5,label=kind)
for label,part in pair_scores[pair_scores.split.eq("train")].groupby("label"):
    axes[1].hist(part.cosine,bins=35,alpha=.5,label=f"label={label}")
axes[0].set_title("Train: gap AP−AN");axes[1].set_title("Train: cosine cặp theo nhãn")
for ax in axes:ax.legend()
fig.tight_layout();fig.savefig(REPORT_DIR / "negative_difficulty.png");plt.show()
# Các ca khó chỉ để rà soát nhãn, không tự sửa.
train_scores=triplet_scores[triplet_scores.split.eq("train")].reset_index(drop=True)
indices=train_scores.nsmallest(10,"gap").index
hard_review=samples[("triplet_data","train")].iloc[indices].copy()
hard_review["gap"]=train_scores.loc[indices,"gap"].to_numpy()
display(hard_review);hard_review.to_csv(REPORT_DIR / "hard_negative_review.csv",index=False)


## 11. Chẩn đoán mô hình có thể học mẹo độ dài — mẫu train/val
Baseline logistic chỉ dùng độ dài hoặc đặc trưng ký tự; fit trên mẫu train và đo trên mẫu validation,
ngưỡng cố định 0,5. Kết quả cao là lý do điều tra thêm bias, không chứng minh model chính đang học mẹo.
Không dùng test để chọn mô hình baseline này, không dùng điểm này thay kết quả model chính.
Đọc balanced accuracy, MCC và recall lớp 0 cùng F1 lớp 1. So baseline đa số trên **cùng mẫu validation**; F1 cao khi đoán hầu hết lớp 1 có thể chỉ phản ánh lệch lớp.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from evaluation.common import binary_metrics
from sklearn.metrics import balanced_accuracy_score, matthews_corrcoef
def diagnostic_metrics(y, probability):
    predicted=np.asarray(probability)>=.5
    labels=np.asarray(y)
    return {**binary_metrics(labels,probability,predicted),
        "balanced_accuracy":float(balanced_accuracy_score(labels,predicted)),
        "mcc":float(matthews_corrcoef(labels,predicted)),
        "recall_class0":float((~predicted[labels==0]).mean()) if (labels==0).any() else None}
train_features=features[features.split.eq("train")];val_features=features[features.split.eq("val")]
shortcut_results=[]
if train_features.label.nunique()==2 and val_features.label.nunique()==2:
    shortcut_results.append({"baseline":"train_majority",**diagnostic_metrics(val_features.label,np.full(len(val_features),majority,dtype=float))})
    for name,columns in {"length_only":["words","characters"],
        "surface_features":["words","characters","digit_ratio","punctuation_ratio","uppercase_ratio","unique_word_ratio"]}.items():
        classifier=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,random_state=SEED))
        classifier.fit(train_features[columns],train_features.label)
        probability=classifier.predict_proba(val_features[columns])[:,1]
        shortcut_results.append({"baseline":name,**diagnostic_metrics(val_features.label,probability)})
    display(pd.DataFrame(shortcut_results))
    (REPORT_DIR / "shortcut_baselines.json").write_text(json.dumps(shortcut_results,indent=2),encoding="utf-8")
else:print("Mẫu thiếu một lớp; không fit baseline.")


## 12. Độ dài token thực — tùy chọn, không tải model
Số từ ở các phần trên không xác định được tỷ lệ bị truncate. Nếu đã có tokenizer local, bật ô này để
đo số token của mẫu: bước 0 (128), tầng 3 từng câu (128), tầng 4 cả cặp (256).
Sửa giới hạn theo cấu hình thí nghiệm; không tự thay đổi max_length dựa vào test.

In [ ]:
RUN_TOKEN_LENGTHS=False
TOKENIZER_DIRS={0: ROOT / "checkpoints/tang0",3: ROOT / "checkpoints/tang3",4: ROOT / "checkpoints/tang4"}
if RUN_TOKEN_LENGTHS:
    from transformers import AutoTokenizer
    token_rows=[]
    for tier,directory in TOKENIZER_DIRS.items():
        tokenizer=AutoTokenizer.from_pretrained(str(directory),local_files_only=True)
        frame=samples[({0:"tang0",3:"triplet_data",4:"pairwise_labeled"}[tier],"train")]
        cols={0:["sentence"],3:["anchor","positive","negative"],4:["sentence_1"]}[tier]
        for col in cols:
            lengths=[]
            for start in range(0,len(frame),128):
                batch=frame.iloc[start:start+128]
                encoded=tokenizer(batch[col].tolist(),text_pair=batch.sentence_2.tolist() if tier==4 else None,
                    truncation=False,padding=False,verbose=False)
                lengths.extend(map(len,encoded["input_ids"]))
            lengths=np.array(lengths);limit=256 if tier==4 else 128
            token_rows.append({"tier":tier,"column":col,"limit":limit,"p95":float(np.quantile(lengths,.95)),
                "max":int(lengths.max()),"fraction_over_limit":float((lengths>limit).mean())})
    display(pd.DataFrame(token_rows));pd.DataFrame(token_rows).to_csv(REPORT_DIR / "sample_token_lengths.csv",index=False)
else:print("Chưa đo token: cần tokenizer local và RUN_TOKEN_LENGTHS=True.")


## 13. Audit rò rỉ toàn bộ text và group — tùy chọn, dùng ổ đĩa
Quét 9 split và Task 4/5, chuẩn hóa NFC + khoảng trắng, phân biệt hoa/thường như audit data prep.
SQLite giảm RAM nhưng cần dung lượng ổ đĩa và thời gian đáng kể. Database mới được tạo trong REPORT_DIR;
không sửa dữ liệu đầu vào. Đây chỉ kiểm tra exact normalized overlap, không phát hiện mọi paraphrase gần trùng.

In [ ]:
RUN_FULL_LEAKAGE_AUDIT=False
if RUN_FULL_LEAKAGE_AUDIT:
    from uuid import uuid4
    audit_specs=[]
    for prefix,cols in specs.items():
        groups_cols={"tang0":["group_id"],"triplet_data":["group_id","negative_group_id"],
                     "pairwise_labeled":["group_id","candidate_group_id"]}[prefix]
        for split in ("train","val","test"):
            audit_specs.append((DATA_DIR / f"{prefix}_{split}.csv",split,cols,groups_cols))
    for task in (4,5):audit_specs.append((DATA_DIR / f"padben_task{task}_test.csv","test",["sentence"],["group_id"]))
    fresh_audit=disk_overlap_audit(audit_specs,REPORT_DIR / f"leakage_{uuid4().hex}.sqlite")
    display(fresh_audit)
    (REPORT_DIR / "fresh_leakage_audit.json").write_text(json.dumps(fresh_audit,indent=2),encoding="utf-8")
else:print("Chưa chạy full audit trong phiên này; audit_report.json phía trên là báo cáo đã lưu.")


## 14. Ghi nhận phát hiện trước khi train
Điền nhận xét dựa trên các bảng, không chỉ ghi “dữ liệu sạch”. Không tự xóa/đổi nhãn trong EDA.

| Hạng mục | Bằng chứng cần ghi | Quyết định |
|---|---|---|
| Chất lượng/nhãn | Số lỗi toàn file, các text bị cách ly | Sửa data prep nếu có lỗi |
| Phân phối | Tỷ lệ nhãn theo nguồn; baseline đa số | Chọn metric phù hợp |
| Rò rỉ | Audit mới hay cũ; full hay sample | Không train nếu xác nhận rò rỉ |
| Độ dài | P95/P99; token truncation nếu đã đo | Chọn max_length trên train/val |
| Negative | Gap theo random/hard, ví dụ đọc tay | Ghi nhận ca chưa chắc nhãn |
| Bias | Baseline độ dài trên validation | Thiết kế kiểm tra bổ sung |

Các bảng/biểu đồ nằm trong REPORT_DIR. Kết quả chỉ có ý nghĩa theo seed và kích thước mẫu đã khai báo.